<a href="https://colab.research.google.com/github/fauzanrahmatullah/information-retrieval/blob/main/crawling-information-retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git config --global user.email "fauzan1464@gmail.com"
!git config --global user.name "fauzanrahmatullah"

In [2]:
import requests
import time
import re
from urllib.robotparser import RobotFileParser
from bs4 import BeautifulSoup
from collections import defaultdict

seed_url = "https://upnjatim.ac.id"
robots_url = f"{seed_url}/robots.txt"

rp = RobotFileParser()
rp.set_url(robots_url)
try:
    rp.read()
    print("Berhasil membaca robots.txt dari UPN Jatim")
except:
    print("Gagal membaca robots.txt, tetap akan menggunakan jeda sopan.")

user_agent = "Fauzan_Informatics_Crawler_Bot/1.0"

# setup frontier

frontier = [seed_url]
visited = set()
documents = {}  # {doc_id: {'url': url, 'judul': judul, 'isi': isi}}
doc_id = 0
batas_maksimal = 200

def ekstrak_isi(html, base_url):
    """
    TUGAS 3: Ekstrak isi (judul, isi utama) dan buang boilerplate (menu, footer)
    """
    soup = BeautifulSoup(html, 'html.parser')

    # pembersihan boilerplate
    for element in soup(["script", "style", "nav", "footer", "header", "aside"]):
        element.decompose()

    # ambil judul
    title = soup.title.string if soup.title else "Tanpa Judul"
    title = title.strip()

    # ambil isi
    paragraphs = soup.find_all('p')
    isi_utama = " ".join([p.get_text(strip=True) for p in paragraphs])

    # ekstrak tautan baru terus tambah ke frontier
    links = []
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        # normalisasi URL
        if href.startswith('/'):
            href = base_url + href
        if href.startswith(base_url):
            links.append(href)

    return title, isi_utama, links

print(f"Memulai crawling maksimal {batas_maksimal} halaman...")
print("-" * 50)

# Loop Crawler
while frontier and doc_id < batas_maksimal:
    url = frontier.pop(0)

    # cek duplikasi URL
    if url in visited:
        continue

    # aturan robots.txt
    if not rp.can_fetch(user_agent, url):
        print(f"[-] Dilewati (Dilarang robots.txt): {url}")
        visited.add(url)
        continue

    print(f"[{doc_id+1}/{batas_maksimal}] Mengunduh: {url}")

    try:
        headers = {'User-Agent': user_agent}
        response = requests.get(url, headers=headers, timeout=10)
        visited.add(url)

        # Hanya proses halaman HTML yang sukses (Kode 200)
        if response.status_code == 200 and 'text/html' in response.headers.get('Content-Type', ''):
            judul, isi, tautan_baru = ekstrak_isi(response.text, seed_url)

            # Simpan jika ada teks isinya
            if isi:
                documents[doc_id] = {
                    'url': url,
                    'judul': judul,
                    'isi': isi
                }
                doc_id += 1

                # Masukkan tautan temuan baru ke antrean (frontier)
                for t in tautan_baru:
                    if t not in visited and t not in frontier:
                        frontier.append(t)

        # jarak 2 detik
        time.sleep(2)

    except Exception as e:
        print(f"[!] Gagal mengunduh {url}: {e}")
        visited.add(url)

print("-" * 50)
print("Crawling selesai!\n")


# INVERTED INDEX (pertemuan 2 & 3)
inverted_index = defaultdict(set)
term_frequencies = defaultdict(lambda: defaultdict(int))

print("membuat inverted index...")
for d_id, data in documents.items():
    # merge judul + isi
    teks_mentah = data['judul'] + " " + data['isi']

    # case folding & buang selain huruf/angka
    teks_bersih = re.sub(r'[^a-z0-9\s]', '', teks_mentah.lower())
    tokens = teks_bersih.split()

    for token in tokens:
        inverted_index[token].add(d_id)
        term_frequencies[d_id][token] += 1

print(f"Total dokumen berhasil dikumpulkan: {len(documents)}")
print(f"Total kosakata (term) unik dalam indeks: {len(inverted_index)}")



Berhasil membaca robots.txt dari UPN Jatim
Memulai crawling maksimal 200 halaman...
--------------------------------------------------
[1/200] Mengunduh: https://upnjatim.ac.id
[2/200] Mengunduh: https://upnjatim.ac.id/
[3/200] Mengunduh: https://upnjatim.ac.id/sejarah/
[4/200] Mengunduh: https://upnjatim.ac.id/visi-misi/
[5/200] Mengunduh: https://upnjatim.ac.id/struktur-organisasi-2/
[5/200] Mengunduh: https://upnjatim.ac.id/kampus-bela-negara/
[6/200] Mengunduh: https://upnjatim.ac.id/sasaran-strategis/
[7/200] Mengunduh: https://upnjatim.ac.id/tujuan/
[8/200] Mengunduh: https://upnjatim.ac.id/hymne-mars/
[9/200] Mengunduh: https://upnjatim.ac.id/lambang-logo/
[9/200] Mengunduh: https://upnjatim.ac.id/tarif-layanan/
[10/200] Mengunduh: https://upnjatim.ac.id/seleksi-masuk/
[11/200] Mengunduh: https://upnjatim.ac.id/akreditasi-instansi-prodi/
[12/200] Mengunduh: https://upnjatim.ac.id/kalender-akademik-ta-2025-2026/
[12/200] Mengunduh: https://upnjatim.ac.id/program-sarjana/
[13/200]

In [3]:
# pencarian kata
kata_target = 'mitra'

if kata_target in inverted_index:
    doc_ids_terkait = list(inverted_index[kata_target])
    print(f"\nContoh Indeks untuk kata '{kata_target}': terdapat di {len(doc_ids_terkait)} dokumen.")

    # print 10 docID pertama
    print(f"Muncul di DocID: {doc_ids_terkait[:10]}")
else:
    print(f"\nKata '{kata_target}' tidak ditemukan di seluruh halaman kampus yang di-crawl.")


Contoh Indeks untuk kata 'mitra': terdapat di 3 dokumen.
Muncul di DocID: [90, 99, 98]
